In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.io as sio
from scipy import stats
import scikits.bootstrap as boot
import seaborn as sns
from sklearn import svm
from sklearn.model_selection import train_test_split,cross_val_score,cross_val_predict,cross_validate
from sklearn.metrics import accuracy_score
from scipy.stats import ttest_rel,wilcoxon
import os,math,mat73,sys,h5py
from functions import mybarplot,fisherztrans,myheatmap
from joblib import Parallel, delayed
from datetime import datetime
from tqdm import tqdm

### set parameters

subjs = ['CN040','CN041']
hemi = ['lh','rh']
tasks = ['face','fixation']
roi_labels = ['V1','V2','V3']

nsubj = len(subjs)
nvoxel = 100
ntrial = 1280
ntrialperpos = 80
npos=16
ds0 = 2 # degree

cm = np.array([[255, 107, 129],[112, 161, 255]])/255

ds_list = np.zeros((npos,npos))
for s1 in range(npos):
    r1 = s1//4
    c1 = s1%4
    for s2 in range(s1):
        r2 = s2//4
        c2 = s2%4
        ds_list[s1,s2] = ((r1-r2)**2+(c1-c2)**2)**0.5*ds0
d = np.unique(ds_list)[1:]

nsample = nsubj * np.max([ds_list[ds_list==dd].shape for dd in d[1:]])

ps_list = np.zeros(npos)
for s in range(npos):
    sr = s//4*2
    sc = s%4*2
    ps_list[s] = ((5-sr)**2+(5-sc)**2)**0.5
posd = np.unique(ps_list)
nsample0 = nsubj * np.max([ps_list[ps_list==pp].shape for pp in posd])


In [ ]:
# ====================================================
# load data
data = np.load('../exampledata/selectdata.npz',allow_pickle=True)
stimposicond_all = data['stimposicond_all']-1 # stimposicond_all: [tasks * trial x sample]
beta_all = data['beta_sel_all'] # beta_sel_all: task x roi x sample x vertex x trial


# segment session:
sess_subj = [8,8,8,8,8,8,8,9]
ntrialperrun = 32
maxrun = 7
beta_session = np.full([len(tasks),len(roi_labels),nsubj,max(sess_subj),nvoxel,ntrialperrun*maxrun],np.nan)
stim_session = np.full([len(tasks),nsubj,max(sess_subj),ntrialperrun*maxrun],np.nan)

RUN_IDX = {
    (4, 0): {'face': [3, 5], 'fixation': [0, 1, 2, 4, 6]},
    (4, 6): {'face': [0, 2, 4, 6, 8, 10, 11], 'fixation': [1, 3, 5, 7, 9]},
    (4, 7): {'face': [0, 2, 4, 6, 8, 10], 'fixation': [1, 3, 5, 7, 9]},
    (7, 0): {'face': [0, 2, 4], 'fixation': [1, 3, 5]},
    (7, 4): {'face': [1], 'fixation': [0,4]},
    (7, 5): {'face': [0, 2, 4, 6, 8, 10], 'fixation': [1, 3, 5, 7, 9, 11]},
    (7, 7): {'face': [0, 2, 4, 6, 8, 10], 'fixation': [1, 3, 5, 7, 9, 11]},
    (7, 8): {'face': [0, 2, 4, 6], 'fixation': [1, 3, 5]},
}
DEFAULT_RUN_IDX = {'face': [0,2,4,6,8], 'fixation': [1,3,5,7,9]}


for task_i in range(len(tasks)):
    for roi_i in range(len(roi_labels)):
        for subj_i in range(nsubj):
            count_trial = 0
            for sess_i in range(sess_subj[subj_i]):
                run_idx = RUN_IDX.get((subj_i, sess_i), DEFAULT_RUN_IDX)
                trial_idx = np.arange(0,len(run_idx[tasks[task_i]])*ntrialperrun)
                beta_session[task_i,roi_i,subj_i,sess_i,:,trial_idx] = beta_all[task_i,roi_i,subj_i,:,count_trial+trial_idx]
                stim_session[task_i,subj_i,sess_i,trial_idx] = stimposicond_all[task_i,subj_i,count_trial+trial_idx]
                count_trial += trial_idx[-1] + 1

###Session 5 of subject CN056 was excluded from analysis due to insufficient runs.
nsess = 8
subj_bad = 7; sess_bad = 4
beta_session[:,:,subj_bad,:nsess] = np.delete(beta_session[:,:,subj_bad],sess_bad,axis=2)
beta_session = beta_session[:,:,:,:nsess]
stim_session[:,subj_bad,:nsess] = np.delete(stim_session[:,subj_bad],sess_bad,axis=1)
stim_session = stim_session[:,:,:nsess]

print(beta_session.shape) # 2 task x 7 roi x 8 subj x 8 session x 100 voxel x 224 trial
print(stim_session.shape) # 2 task x 8 subj x 8 session x 224 trial


# ====================================================
# load glm r2

data = np.load('../exampledata.npz',allow_pickle=True)
v_sel_all = data['v_sel_all'] # roi x subj x voxel

glm_r2_all = np.full([len(subjs),len(roi_labels),nvoxel,9],np.nan) # subj x roi x voxel x session
for subj_i in range(len(subjs)):
    
    # The raw data is available upon reasonable request.

    os.chdir('/home/data/rawdata/facePRF/'+subjs[subj_i]+'_faceprf/faceprfanalyze_vol')
    with h5py.File(subjs[subj_i] + '_PRFdatasets.mat', 'r') as f:
        R2 = np.array(f['data']['R2']).T

    for roi_i in range(len(roi_labels)):
        vind = v_sel_all[roi_i,subj_i]
        glm_r2 = R2[vind,:]
        glm_r2_all[subj_i,roi_i,:,:glm_r2.shape[1]] = glm_r2

nsess = 8
subj_bad = 7; sess_bad = 4
glm_r2_all[subj_bad,:,:,:nsess] = np.delete(glm_r2_all[subj_bad,:],sess_bad,axis=2)
glm_r2_all = glm_r2_all[:,:,:,:nsess]


np.savez('.../exampledata/driftcheck.npz',glm_r2_all=glm_r2_all,beta_session=beta_session,stim_session=stim_session)
